In [ ]:
# ----------- Import podstawowych bibliotek ROS2 -----------
# Główna biblioteka ROS2 dla Pythona – pozwala na uruchamianie i obsługę nodów (programów w ROS2)
import rclpy 

# Klasa bazowa do tworzenia własnych nodów – każdy node w ROS2 dziedziczy po tej klasie
from rclpy.node import Node

# Wiadomość ROS2 używana do przekazywania prędkości liniowych i kątowych robota (np. do sterowania ruchem)
from geometry_msgs.msg import Twist

# ----------- Import podstawowych bibliotek Python -----------
# Biblioteka do tworzenia interaktywnych elementów (np. przycisków, pól tekstowych) w Jupyter Notebook.  
import ipywidgets as widgets          

# Funkcje z IPython do wyświetlania widgetów w notebooku oraz czyszczenia poprzedniego outputu przed 
# wyświetleniem nowych wartości.  
from IPython.display import display, clear_output  

# Biblioteka do uruchamiania zadań w osobnych wątkach, np. ciągła publikacja wiadomości 
# równolegle z obsługą interfejsu.  
import threading     

# Biblioteka do obsługi czasu, tutaj używana do kontrolowania częstotliwości publikacji wiadomości.  
import time                           



# --- Definicja klasy VelocityPublisher ---
class VelocityPublisher(Node):
    def __init__(self):
        # Inicjalizacja węzła ROS2 o nazwie 'velocity_publisher'
        super().__init__('velocity_publisher')
        
        # Publisher, który wysyła wiadomości typu Twist na temat /cmd_vel
        self.publisher_ = self.create_publisher(Twist, '/cmd_vel', 10)
        
        # Obiekt wiadomości Twist przechowujący aktualne prędkości robota
        self.twist = Twist()
        
        # Pole output do wyświetlania informacji tekstowych w notebooku
        self.output = widgets.Output()
        
        # Flaga sterująca pracą pętli publikującej
        self.running = True
        
        # --- Widgety do wyświetlania aktualnych wartości prędkości ---
        self.linear_x_display = widgets.FloatText(value=0.0, description='Linear X:', disabled=True)
        self.linear_y_display = widgets.FloatText(value=0.0, description='Linear Y:', disabled=True)
        self.angular_z_display = widgets.FloatText(value=0.0, description='Angular Z:', disabled=True)
        
        # --- Widget do ustawienia częstotliwości publikacji ---
        self.freq_label = widgets.Label(value='Topic freq:')
        self.freq_box = widgets.FloatText(value=10.0, description='Hz')
        
        # --- Widgety do ustawienia kroków zmiany prędkości ---
        self.step_label = widgets.Label(value='Step sizes:')
        self.linear_step_box = widgets.FloatText(value=0.05, description='Linear Step:')
        self.angular_step_box = widgets.FloatText(value=0.01, description='Angular Step:')
        
        # --- Przycisk resetujący prędkości do zera ---
        self.zero_button = widgets.Button(description='Zero Velocity')
        self.zero_button.on_click(self.zero_velocity)  # Po kliknięciu wywołana zostanie metoda zero_velocity()
        
        # Wyświetlenie widgetów w notebooku
        display(self.linear_x_display, self.linear_y_display, self.angular_z_display)
        display(widgets.HBox([self.freq_label, self.freq_box]))
        display(widgets.HBox([self.step_label, self.linear_step_box, self.angular_step_box]))
        display(self.zero_button)
        
        # --- Uruchomienie osobnego wątku do publikacji prędkości ---
        # Dzięki temu program wciąż reaguje na przyciski w notebooku
        self.publish_thread = threading.Thread(target=self.publish_velocity, daemon=True)
        self.publish_thread.start()
        
    def update_velocity(self, linear_x=0.0, linear_y=0.0, angular_z=0.0):
        """
        Aktualizuje prędkość robota (w wiadomości Twist) na podstawie zmian zadanych przez użytkownika.
        """
        # Dodanie wartości do aktualnych prędkości
        self.twist.linear.x += linear_x
        self.twist.linear.y += linear_y
        self.twist.angular.z += angular_z
        
        # Aktualizacja wartości w polach tekstowych
        self.linear_x_display.value = self.twist.linear.x
        self.linear_y_display.value = self.twist.linear.y
        self.angular_z_display.value = self.twist.angular.z
        
        # Wyświetlenie nowych wartości w output (tekstowym oknie w notebooku)
        with self.output:
            clear_output(wait=True)
            print(f'Current Velocity:\nLinear: x={self.twist.linear.x}, y={self.twist.linear.y}\nAngular: z={self.twist.angular.z}')
        
    def zero_velocity(self, _):
        """
        Resetuje prędkość robota do zera (Twist = 0).
        """
        self.twist = Twist()
        self.update_velocity()   # Aktualizacja wyświetlanych wartości
        self.get_logger().info('Velocity reset to zero.')  # Log w konsoli ROS2
        
    def publish_velocity(self):
        """
        Pętla publikująca wiadomości Twist na temat /cmd_vel z zadaną częstotliwością.
        """
        while self.running:
            self.publisher_.publish(self.twist)  # Publikacja aktualnej wiadomości
            time.sleep(1.0 / max(1.0, self.freq_box.value))  # Odstęp czasu zależny od częstotliwości (Hz)
        
# --- Inicjalizacja ROS2 ---
rclpy.init()
node = VelocityPublisher()

# --- Uruchomienie spina w osobnym wątku ---
# Dzięki temu ROS2 obsługuje callbacki, a notebook pozostaje responsywny
def ros_spin():
    rclpy.spin(node)

spin_thread = threading.Thread(target=ros_spin, daemon=True)
spin_thread.start()

# --- Definicja przycisków sterujących ---
button_x_plus = widgets.Button(description='X +')
button_x_minus = widgets.Button(description='X -')
button_y_plus = widgets.Button(description='Y +')
button_y_minus = widgets.Button(description='Y -')
button_angular_plus = widgets.Button(description='Yaw +')
button_angular_minus = widgets.Button(description='Yaw -')

# --- Przypisanie akcji do przycisków ---
# Po kliknięciu dany przycisk zmienia prędkość robota o wartość zdefiniowaną w polach step_box
button_x_plus.on_click(lambda b: node.update_velocity(linear_x=node.linear_step_box.value))
button_x_minus.on_click(lambda b: node.update_velocity(linear_x=-node.linear_step_box.value))
button_y_plus.on_click(lambda b: node.update_velocity(linear_y=node.linear_step_box.value))
button_y_minus.on_click(lambda b: node.update_velocity(linear_y=-node.linear_step_box.value))
button_angular_plus.on_click(lambda b: node.update_velocity(angular_z=node.angular_step_box.value))
button_angular_minus.on_click(lambda b: node.update_velocity(angular_z=-node.angular_step_box.value))

# --- Wyświetlenie przycisków w notebooku ---
display(widgets.HBox([button_x_minus, button_x_plus]))
display(widgets.HBox([button_y_minus, button_y_plus]))
display(widgets.HBox([button_angular_minus, button_angular_plus]))
display(node.output)
